# S11 pratimas

```
S11 pratimas -- sintezė ir temų modeliavimas.

Užduotis
--------
(a) Įgyvendinkite ``run_topics(texts, embeddings)`` -- BERTopic su iš anksto
    apskaičiuotais įterpiniais, fiksuota UMAP sėkla ir konfigūruotu HDBSCAN.
    ``bertopic``/``umap``/``hdbscan`` importuokite TIK funkcijos viduje.
    Paleiskite ant 12 dokumentų korpuso ir palyginkite rastų temų skaičių su
    projektuotomis 3 temomis.
(b) Įgyvendinkite ``map_study(study_id, rq, charted, *, call_llm)`` --
    ``study_id`` imkite IŠ PARAMETRO, ne iš modelio atsakymo; iškelkite
    ``ValueError``, jei ``cited_fields`` tuščias arba nurodo nesamą lauką.
    Tada ``refine_synthesis(rq, answers, ...)`` -- reikalaukite TIKSLIOS
    studijų aprėpties ir iškelkite A-vs-B prieštarą.

Paleidimas: ``python -m examples.chapter.s11_synthesis_topics.exercise``
```

1. Paleiskite paruošimo langelį. 2. Užpildykite `TODO` funkcijas pratimo langelyje ir paleiskite jį. 3. Palyginkite su pavyzdiniu sprendimu.

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/rimraul/slr-automation.git"
REPO_DIR = REPO_URL.rsplit("/", 1)[-1].removesuffix(".git")
if "google.colab" in sys.modules and not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
for req in ['examples/chapter/requirements-colab.txt', 'examples/chapter/requirements-colab-topics.txt']:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit("pip install nepavyko (zr. klaida aukščiau): " + req)
print("Paruosta:", os.getcwd())

In [ ]:

from __future__ import annotations

from pydantic import BaseModel

from examples.chapter import toolkit


class StudyAnswer(BaseModel):
    study_id: str
    answer_text: str
    cited_fields: list[str]


class Synthesis(BaseModel):
    narrative: str
    cited_study_ids: list[str]
    contradictions: list[str]


class TopicResult(BaseModel):
    assignments: list[int]
    topic_terms: dict[int, list[str]]


def run_topics(texts: list[str], embeddings) -> TopicResult:
    """TODO: BERTopic su iš anksto apskaičiuotais įterpiniais (lazy import)."""
    raise NotImplementedError


def map_study(study_id: str, rq: str, charted: dict, *, call_llm=toolkit.call_llm) -> StudyAnswer:
    """TODO: viena studija, tapatybė iš parametro."""
    raise NotImplementedError


def refine_synthesis(rq: str, answers, *, batch_size: int = 2, call_llm=toolkit.call_llm) -> Synthesis:
    """TODO: sulieti atsakymus, reikalauti tikslios aprėpties."""
    raise NotImplementedError


def main() -> None:
    data = toolkit.load_fixture("synthesis_studies.json")
    s11 = toolkit.SCRIPTED_RESPONSES["s11"]
    map_fake = toolkit.scripted_llm(
        s11["map"], key_fn=lambda m: m[-1]["content"].split("STUDY_ID: ", 1)[1].split("\n", 1)[0]
    )
    print("S11 synthesis exercise")
    try:
        answers = [
            map_study(s["study_id"], data["rq"], s["charted"], call_llm=map_fake)
            for s in data["studies"]
        ]
    except NotImplementedError:
        print("map_study() is not implemented yet -- fill it in and re-run.")
        return
    print(f"mapped {len(answers)} studies")


main()

## Pavyzdinis sprendimas

Pažiūrėkite tik pabandę patys. Žemiau esantys testai tikrina **pavyzdinį sprendimą**, ne jūsų kodą.

In [ ]:
!cat examples/chapter/s11_synthesis_topics/solution.py

In [ ]:
!python -m pytest examples/chapter/s11_synthesis_topics -q

In [ ]:
# NEPRIVALOMA: tikri kreipiniai i DKM (sumoka uz naudojimą). Raktą laikykite
# Colab "Secrets" (rakto piktograma kairėje), niekada ne užrašykite kode.
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")